# NLP Mastery Journey — Module 10: Summarization, Translation, QA & Evaluating Generated Text

This module covers the three classic "generation" tasks — summarization, translation, question answering — and then does something every earlier module has been building toward: **how do you actually measure whether generated text is any good?** Every module so far had a clean accuracy/F1 number to check against. Generated text doesn't — "how good is this summary?" has no single right answer, which is exactly why this needed its own dedicated treatment.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | Extractive summarization, from scratch | A real, working summarizer using only TF-IDF (Module 3) |
| 2 | Abstractive summarization | The pretrained-model approach, and how it differs |
| 3 | Machine translation | Encoder-decoder Transformers (Module 7) applied to translation |
| 4 | Question answering | Extractive QA (span-finding) vs. generative QA |
| 5 | BLEU, from scratch | The classic translation/generation metric — the actual math, not just a library call |
| 6 | ROUGE, from scratch | The classic summarization metric — ROUGE-N and ROUGE-L |
| 7 | BERTScore & perplexity | Semantic-similarity-based and probability-based metrics |
| 8 | LLM-as-judge | The metric most production teams actually lean on today |
| 9 | Choosing the right metric | A decision guide — different tasks need different metrics |
| 10 | Production evaluation pipelines | Regression testing generation quality over time |

### How to use this notebook
- **BLEU and ROUGE are implemented completely from scratch and run live** — you compute real scores on real example text, no libraries needed.
- The extractive summarizer in Part 1 also runs live, using TF-IDF you already know from Module 3.
- Anything needing a pretrained model (abstractive summarization, translation, extractive QA, BERTScore) is shown as a correct, ready-to-use template, commented out — same pattern as every module since Module 3.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.


## 0. Setup

In [ ]:
# Parts 1 and 5-6 (extractive summarization, BLEU, ROUGE) need nothing new -
# just numpy, pandas, and scikit-learn, which you already have.
#
# For the PRETRAINED-model sections (commented out), you would run:
# %pip install transformers torch rouge-score sacrebleu bert-score

import numpy as np
import pandas as pd

print("Setup done. No installs needed for the live parts of this notebook.")


## Part 1 — Extractive Summarization, From Scratch

Two families of summarization:
- **Extractive**: pick out the most important SENTENCES that already exist in the document, and stitch them together as the summary. Simpler, always factually grounded in the source (it's literally copied from it), but can read a bit choppy.
- **Abstractive**: generate NEW sentences that capture the meaning, potentially using words that never appeared in the original. Reads more naturally, but can hallucinate details that aren't actually in the source.

We'll build a real, working extractive summarizer using an idea close to the classic **TextRank** algorithm: score each sentence by how SIMILAR it is to the rest of the document (using TF-IDF + cosine similarity, exactly like Module 3/9), on the idea that a sentence central to the document's overall content is more likely to be summary-worthy.


In [ ]:
example_article = """
Renewable energy use has grown rapidly over the past decade.
Solar and wind power costs have fallen sharply, making them competitive with fossil fuels.
Many countries have set targets to reduce carbon emissions by switching to renewable sources.
However, storing renewable energy remains a technical challenge, since the sun does not always shine and the wind does not always blow.
Battery technology has improved, but large-scale storage is still expensive.
Some experts believe that improvements in battery storage will be the deciding factor for how fast the transition to renewables happens.
Governments are investing heavily in both renewable generation and storage research.
"""

# ── Step 1: split the article into individual sentences ─────────────────────
def split_into_sentences(text):
    raw_sentences = text.split(".")
    sentences = []
    for sentence in raw_sentences:
        cleaned = sentence.strip()
        if cleaned:
            sentences.append(cleaned + ".")
    return sentences

sentences = split_into_sentences(example_article)
print(f"Number of sentences: {len(sentences)}")
for i, s in enumerate(sentences):
    print(f"  {i}: {s}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def extractive_summarize(sentences, num_sentences=3):
    """
    📋 COPY-PASTE TEMPLATE
    Step-by-step:
    1. Turn every sentence into a TF-IDF vector.
    2. For each sentence, measure how similar it is to EVERY other sentence,
       and add those similarities up -> a sentence "importance" score.
       (A sentence that overlaps a lot with the rest of the document is
       treated as more central / more summary-worthy.)
    3. Pick the `num_sentences` highest-scoring sentences.
    4. Put them back in their ORIGINAL order, so the summary still reads
       in a natural, chronological order.
    """
    # Step 1: vectorize every sentence
    vectorizer = TfidfVectorizer()
    sentence_vectors = vectorizer.fit_transform(sentences)

    # Step 2: build a full sentence-to-sentence similarity matrix, then sum each row
    similarity_matrix = cosine_similarity(sentence_vectors)
    importance_scores = similarity_matrix.sum(axis=1)

    # Step 3: find the indices of the top-scoring sentences
    top_sentence_indices = np.argsort(importance_scores)[::-1][:num_sentences]

    # Step 4: sort those indices back into original document order
    top_sentence_indices_in_order = sorted(top_sentence_indices)

    summary_sentences = [sentences[i] for i in top_sentence_indices_in_order]
    return " ".join(summary_sentences), importance_scores

summary, scores = extractive_summarize(sentences, num_sentences=3)

print("Sentence importance scores:")
for i, (sentence, score) in enumerate(zip(sentences, scores)):
    print(f"  {i}: {score:.3f}  {sentence}")

print()
print("SUMMARY:")
print(summary)


In [ ]:
# 🔀 Extractive vs abstractive - when to reach for which
# | Situation                                            | Choose               |
# |------------------------------------------------------------|------------------------|
# | Need a FAST, cheap, always-factually-grounded summary          | Extractive (what we built above)|
# | Need a SHORT, natural-reading summary; some cost/latency is OK  | Abstractive (below)   |
# | Summarizing something legally/factually sensitive (need to be able|
# | to point to EXACTLY which sentence a claim came from)             | Extractive           |

# 📋 COPY-PASTE TEMPLATE - abstractive summarization with a pretrained model:
# from transformers import pipeline
# summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
# result = summarizer(example_article, max_length=60, min_length=20, do_sample=False)
# print(result[0]["summary_text"])

print("Abstractive summarization template shown above - needs internet + transformers.")


## Part 2 — Machine Translation

Translation is the ORIGINAL task the Transformer architecture (Module 7) was built for — an **encoder-decoder** model: the encoder reads the source-language sentence fully (bidirectional), and the decoder generates the target-language sentence one token at a time, using **cross-attention** to look back at the encoder's output at every generation step (deciding which source words are relevant to the word it's generating right now).


In [ ]:
# from transformers import MarianMTModel, MarianTokenizer
#
# model_name = "Helsinki-NLP/opus-mt-en-fr"   # English -> French; Helsinki-NLP
#                                              # publishes hundreds of language-pair models
# tokenizer = MarianTokenizer.from_pretrained(model_name)
# model = MarianMTModel.from_pretrained(model_name)
#
# text_to_translate = "Machine learning models can now translate text remarkably well."
# inputs = tokenizer(text_to_translate, return_tensors="pt", padding=True)
#
# translated_tokens = model.generate(**inputs)
# translation = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
# print(translation)

print("Machine translation template shown above - needs internet + transformers.")

# 🔀 Translation model choices
# | Model family              | Good for...                                          |
# |---------------------------------|-----------------------------------------------------------|
# | Helsinki-NLP/opus-mt-*            | A specific, well-tested language PAIR (e.g. en->fr)          |
# | facebook/nllb-200                   | Very broad language coverage (200+ languages), one model      |
# | A general LLM (few-shot prompted)     | Flexible, good for informal/creative text, but not purpose-tuned|


## Part 3 — Question Answering

Two distinct shapes of "question answering," easy to conflate:

- **Extractive QA**: given a QUESTION and a CONTEXT passage, find the exact SPAN of text within the context that answers it (predicting a start and end position). This is what the classic SQuAD benchmark measures, and it's essentially a specialized token-classification task (Module 6, Part 7 / Module 8, Part 9).
- **Generative (open-domain) QA**: the model WRITES an answer in its own words, optionally using retrieval (Module 9's RAG) to find relevant context first, rather than being handed a single fixed passage.


In [ ]:
# ── Extractive QA ─────────────────────────────────────────────────────────
# from transformers import pipeline
# qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
#
# context = """Photosynthesis is the process plants use to turn sunlight into
# energy. It happens mainly in the leaves, inside structures called chloroplasts."""
# question = "Where does photosynthesis mainly happen?"
#
# result = qa_pipeline(question=question, context=context)
# print(result)
# # e.g. {'answer': 'the leaves', 'score': 0.87, 'start': 71, 'end': 81} -
# # notice it returns the EXACT SPAN and its position, not a freely-written answer

print("Extractive QA pipeline shown above - a direct application of Module 6/8's token classification.")

# ── Generative QA, built from what you ALREADY have (Module 9's RAG) ────────
# This is literally just answer_question_with_rag() from Module 9, Part 10 -
# question answering and RAG are, in practice, often the exact same system.
print("Generative QA = Module 9's RAG pipeline, applied to a question-answering use case.")


## Part 4 — Why Evaluating Generated Text Is Genuinely Hard

Every earlier module had a clean way to check correctness: a predicted LABEL is either right or wrong (Module 5), a predicted TOKEN either matches the true tag or doesn't (Module 6/8). Generated text has **no single correct answer** — there are countless valid ways to summarize the same article or translate the same sentence. The rest of this module covers the different families of metrics built to handle that, and — importantly — none of them is perfect; using the RIGHT one for your task, and knowing its blind spots, matters as much as computing it correctly.


## Part 5 — BLEU, From Scratch

**BLEU** (Bilingual Evaluation Understudy) is the classic machine-translation metric, also used more broadly for other generation tasks. Core idea: compare N-GRAMS (Module 3) in the generated text against N-grams in one or more human-written REFERENCE texts — the more overlap, the higher the score.


In [ ]:
from collections import Counter

def get_ngrams(tokens, n):
    """Return a list of all n-grams (as tuples) from a list of tokens."""
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i + n])
        ngrams.append(ngram)
    return ngrams

def ngram_precision(candidate_tokens, reference_tokens, n):
    """
    What fraction of the CANDIDATE's n-grams also appear in the REFERENCE?
    Uses "clipping" - an n-gram can only be counted as a match up to the
    number of times it ACTUALLY appears in the reference, so a candidate
    can't get credit for repeating one good word over and over.
    """
    candidate_ngrams = Counter(get_ngrams(candidate_tokens, n))
    reference_ngrams = Counter(get_ngrams(reference_tokens, n))

    total_matches = 0
    for ngram, count_in_candidate in candidate_ngrams.items():
        count_in_reference = reference_ngrams.get(ngram, 0)
        # "clip" - never count more matches than the reference actually has
        total_matches += min(count_in_candidate, count_in_reference)

    total_candidate_ngrams = sum(candidate_ngrams.values())
    if total_candidate_ngrams == 0:
        return 0.0
    return total_matches / total_candidate_ngrams

# ── Quick check on a simple example ──────────────────────────────────────────
candidate = "the cat sat on the mat".split()
reference = "the cat is sitting on the mat".split()

for n in [1, 2, 3]:
    precision = ngram_precision(candidate, reference, n)
    print(f"{n}-gram precision: {precision:.3f}")


In [ ]:
import math

def compute_bleu(candidate_tokens, reference_tokens, max_n=4):
    """
    📋 COPY-PASTE TEMPLATE
    Step-by-step:
    1. Compute n-gram precision for n = 1, 2, 3, 4.
    2. Combine them with a GEOMETRIC MEAN (multiply, then take the n-th root) -
       this means BLEU rewards candidates that are good at EVERY n-gram size,
       not just good unigram overlap while getting word order totally wrong.
    3. Apply a BREVITY PENALTY - without this, a very SHORT candidate could
       score unfairly well (fewer n-grams = fewer chances to be wrong), so we
       penalize candidates that are much shorter than the reference.
    """
    # Step 1: get precision at each n-gram size
    precisions = []
    for n in range(1, max_n + 1):
        p = ngram_precision(candidate_tokens, reference_tokens, n)
        precisions.append(p)

    # If ANY precision is exactly 0, the geometric mean would be 0 too -
    # in practice this happens a lot on short examples like ours; real BLEU
    # implementations use "smoothing" to avoid a harsh all-or-nothing score.
    # We add a tiny smoothing value here for the same reason:
    smoothed_precisions = [p if p > 0 else 1e-9 for p in precisions]

    # Step 2: geometric mean of the n-gram precisions
    log_precisions = [math.log(p) for p in smoothed_precisions]
    geometric_mean = math.exp(sum(log_precisions) / len(log_precisions))

    # Step 3: brevity penalty
    candidate_length = len(candidate_tokens)
    reference_length = len(reference_tokens)
    if candidate_length >= reference_length:
        brevity_penalty = 1.0   # no penalty if candidate is as long or longer
    else:
        brevity_penalty = math.exp(1 - reference_length / candidate_length)

    bleu_score = brevity_penalty * geometric_mean
    return bleu_score

bleu = compute_bleu(candidate, reference)
print(f"BLEU score: {bleu:.3f}")

# Try it on a near-perfect match, to sanity-check the math:
near_perfect_candidate = "the cat sat on the mat".split()
near_perfect_reference = "the cat sat on the mat".split()
print(f"BLEU on an EXACT match: {compute_bleu(near_perfect_candidate, near_perfect_reference):.3f}")


In [ ]:
# 📋 COPY-PASTE TEMPLATE - the real library, for production use
# (our from-scratch version is simplified for teaching; sacrebleu is the
# standard, carefully-validated implementation used for actually reporting
# results, since subtle tokenization differences can shift scores):
#
# import sacrebleu
# score = sacrebleu.corpus_bleu(
#     hypotheses,      # list of generated translations
#     [references],    # list of lists - supports MULTIPLE valid reference translations per example
# )
# print(score.score)

print("Real sacrebleu usage shown above - use this, not the from-scratch version, "
      "whenever you need to report a comparable, standard BLEU score.")


## Part 6 — ROUGE, From Scratch

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard SUMMARIZATION metric. The key conceptual difference from BLEU: ROUGE emphasizes **recall** (did the summary capture what the reference says?) over precision (BLEU's focus is more precision-oriented), which fits summarization's actual goal — did you capture the important content, not just avoid adding extra words.

- **ROUGE-N**: n-gram overlap, like BLEU, but computed from the REFERENCE's perspective (recall) rather than the candidate's (precision).
- **ROUGE-L**: based on the **Longest Common Subsequence (LCS)** between candidate and reference — words that appear in the same relative order in both, not necessarily contiguous. This rewards preserving the reference's overall structure/order, even with some words inserted in between.


In [ ]:
def rouge_n_recall(candidate_tokens, reference_tokens, n):
    """
    What fraction of the REFERENCE's n-grams were also produced by the candidate?
    This is the RECALL version - the mirror image of BLEU's precision.
    """
    candidate_ngrams = Counter(get_ngrams(candidate_tokens, n))
    reference_ngrams = Counter(get_ngrams(reference_tokens, n))

    total_matches = 0
    for ngram, count_in_reference in reference_ngrams.items():
        count_in_candidate = candidate_ngrams.get(ngram, 0)
        total_matches += min(count_in_candidate, count_in_reference)

    total_reference_ngrams = sum(reference_ngrams.values())
    if total_reference_ngrams == 0:
        return 0.0
    return total_matches / total_reference_ngrams

rouge_1 = rouge_n_recall(candidate, reference, n=1)
rouge_2 = rouge_n_recall(candidate, reference, n=2)
print(f"ROUGE-1 (unigram recall): {rouge_1:.3f}")
print(f"ROUGE-2 (bigram recall):  {rouge_2:.3f}")


In [ ]:
def longest_common_subsequence_length(tokens_a, tokens_b):
    """
    Classic dynamic-programming LCS length calculation. `table[i][j]` holds
    the LCS length using the first `i` tokens of tokens_a and the first `j`
    tokens of tokens_b - build it up one small piece at a time.
    """
    num_rows = len(tokens_a) + 1
    num_cols = len(tokens_b) + 1
    table = [[0] * num_cols for _ in range(num_rows)]

    for i in range(1, num_rows):
        for j in range(1, num_cols):
            if tokens_a[i - 1] == tokens_b[j - 1]:
                # the tokens match - extend the LCS found so far by 1
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                # they don't match - take whichever previous LCS was longer
                table[i][j] = max(table[i - 1][j], table[i][j - 1])

    return table[num_rows - 1][num_cols - 1]

def rouge_l(candidate_tokens, reference_tokens):
    """
    ROUGE-L combines precision and recall (both based on the LCS length)
    into a single F1-style score - the same balancing idea as Module 5's F1.
    """
    lcs_length = longest_common_subsequence_length(candidate_tokens, reference_tokens)

    precision = lcs_length / len(candidate_tokens) if candidate_tokens else 0.0
    recall = lcs_length / len(reference_tokens) if reference_tokens else 0.0

    if precision + recall == 0:
        return 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return f1

rouge_l_score = rouge_l(candidate, reference)
print(f"ROUGE-L: {rouge_l_score:.3f}")

# ── Trying it on our actual extractive summary from Part 1 ──────────────────
reference_summary = (
    "Renewable energy has grown rapidly as solar and wind costs have fallen. "
    "Storing renewable energy remains a technical challenge. "
    "Battery storage improvements may decide how fast the transition happens."
)
generated_summary_tokens = summary.lower().split()
reference_summary_tokens = reference_summary.lower().split()

print()
print("ROUGE-1:", round(rouge_n_recall(generated_summary_tokens, reference_summary_tokens, 1), 3))
print("ROUGE-L:", round(rouge_l(generated_summary_tokens, reference_summary_tokens), 3))


In [ ]:
# 📋 COPY-PASTE TEMPLATE - the real library:
#
# from rouge_score import rouge_scorer
# scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
# scores = scorer.score(reference_summary, summary)
# print(scores)
# # {'rouge1': Score(precision=..., recall=..., fmeasure=...), 'rouge2': ..., 'rougeL': ...}

print("Real rouge-score library usage shown above.")

# 🔀 BLEU vs ROUGE - which for which task
# | Metric  | Typically used for...        | Emphasizes         |
# |------------|----------------------------------|-------------------------|
# | BLEU         | Machine translation                 | Precision                 |
# | ROUGE          | Summarization                          | Recall                    |
# In practice both get reported for BOTH task types fairly often - they
# capture slightly different things and are cheap to compute together.


## Part 7 — BERTScore & Perplexity

### BERTScore: fixing BLEU/ROUGE's biggest blind spot
BLEU and ROUGE only count EXACT word matches. If your candidate says "the film was fantastic" and the reference says "the movie was great" — ZERO word overlap on the key words, despite being nearly the same meaning. **BERTScore** fixes this by comparing contextual EMBEDDINGS (Module 7) of each word instead of exact strings, so semantically similar-but-differently-worded text still scores well.


In [ ]:
# We don't have a real contextual embedding model available live in this
# notebook, so here's a SIMPLIFIED, TF-IDF-based approximation of the same
# IDEA, purely to make the concept concrete - a real BERTScore uses proper
# contextual embeddings (Module 7), not TF-IDF, and is meaningfully better
# at capturing paraphrases than what's shown here.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def approximate_semantic_similarity(text_a, text_b):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([text_a, text_b])
    similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
    return similarity

exact_overlap_pair = ("the movie was great", "the film was great")
paraphrase_pair = ("the movie was great", "the film was fantastic")

print("Similarity (some shared words):     ", round(approximate_semantic_similarity(*exact_overlap_pair), 3))
print("Similarity (fully different words):  ", round(approximate_semantic_similarity(*paraphrase_pair), 3))
print()
print("Notice the second pair scores much lower here, EVEN THOUGH the meaning is nearly identical -")
print("this is exactly the gap a REAL BERTScore (using contextual embeddings) is built to close.")


In [ ]:
# 📋 COPY-PASTE TEMPLATE - real BERTScore:
#
# from bert_score import score
# precision, recall, f1 = score([candidate_summary], [reference_summary], lang="en")
# print(f"BERTScore F1: {f1.item():.3f}")

print("Real bert-score library usage shown above - needs internet to download its underlying model.")


### Perplexity: measuring how "surprised" a language model is by text
Perplexity measures how well a language model PREDICTS a piece of text — lower perplexity means the model found the text more expected/fluent. It's computed from the model's own token-by-token probabilities:

$$\text{Perplexity} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right)$$

Useful for comparing LANGUAGE MODELS against each other, or flagging unusually disfluent generated text — but it says nothing about whether generated text is factually correct, relevant, or well-formatted, which is why it's rarely used alone as a generation-quality metric.


## Part 8 — LLM-as-Judge: What Most Production Teams Actually Use Today

Every metric above has real, documented blind spots — BLEU/ROUGE miss paraphrases, BERTScore is still an approximation of "good," perplexity ignores correctness entirely. The approach that has become the most common in production over the last couple of years: **ask a strong LLM to evaluate the output directly**, given clear scoring criteria.


In [ ]:
# 📋 COPY-PASTE TEMPLATE - a simple LLM-as-judge prompt

def build_judge_prompt(question, generated_answer, reference_answer=None):
    if reference_answer:
        return f"""You are evaluating the quality of an AI-generated answer.

Question: {question}

Reference (correct) answer: {reference_answer}

Generated answer: {generated_answer}

On a scale of 1-5, how well does the generated answer match the reference
in terms of factual correctness and completeness? Respond with ONLY a
number from 1 to 5, followed by a one-sentence explanation."""
    else:
        return f"""You are evaluating the quality of an AI-generated answer.

Question: {question}

Generated answer: {generated_answer}

Rate this answer from 1-5 on: (a) relevance to the question, (b) clarity,
and (c) whether it seems factually plausible. Respond with ONLY a number
from 1 to 5, followed by a one-sentence explanation."""

judge_prompt = build_judge_prompt(
    question="What does chlorophyll do?",
    generated_answer="Chlorophyll absorbs sunlight, which the plant uses for photosynthesis.",
    reference_answer="Chlorophyll, the green pigment in chloroplasts, absorbs sunlight.",
)
print(judge_prompt)

# This prompt would then be sent to an LLM (same pattern as Module 9, Part 5):
# response = client.messages.create(model="claude-sonnet-5", max_tokens=100,
#                                    messages=[{"role": "user", "content": judge_prompt}])


### 🔀 LLM-as-judge: strengths, and the honest caveats
- **Strength**: captures nuance, paraphrase-tolerance, and task-specific correctness that BLEU/ROUGE simply can't.
- **Caveat 1 — cost/latency**: an extra LLM call per evaluation, which adds up at scale (this is genuinely expensive to run on every single production request; usually reserved for evaluation/monitoring pipelines, not real-time use).
- **Caveat 2 — judge bias**: LLM judges have documented biases (e.g. sometimes preferring longer answers, or answers stylistically similar to their own outputs) — worth spot-checking judge scores against human judgment, especially when first setting one up.
- **Caveat 3 — still not a silver bullet**: for genuinely high-stakes evaluation, human evaluation (with clear rubrics, multiple raters, and measured inter-rater agreement) remains the gold standard that LLM-as-judge is usually validated against, not a full replacement for.


## Part 9 — Choosing the Right Metric for the Right Task

| Task | Primary metric(s) | Why |
|------|----------------------|-------|
| Machine translation | BLEU (or the more modern chrF/COMET) | Precision-oriented n-gram overlap fits "did we produce the right words" |
| Summarization | ROUGE-1/2/L | Recall-oriented fits "did we capture the key content" |
| Paraphrase-heavy tasks (any task where wording legitimately varies a lot) | BERTScore or LLM-as-judge | Exact n-gram overlap under-counts valid paraphrases |
| Open-ended generation (chat, creative writing) | LLM-as-judge + human evaluation | No fixed reference exists to compare n-grams against at all |
| Extractive QA | Exact Match (EM) + F1 on the answer SPAN | The answer is a specific span, not a whole free-form generation |
| Comparing language models generally | Perplexity (on a fixed, standard benchmark corpus) | Measures raw language modeling fluency, not task performance |

**The one-sentence version**: automated n-gram metrics (BLEU/ROUGE) are cheap, fast, and fine for tracking RELATIVE progress during development (did this change make the model better or worse), but for anything you're about to ship, pair them with LLM-as-judge and/or a real human evaluation pass before trusting the number.


## Part 10 — Production: Evaluation Pipelines & Regression Testing

The same discipline that applies to code (unit tests, CI) applies to generation quality:

- **A fixed evaluation set**: a set of inputs with known-good reference outputs (or clear grading criteria) that NEVER changes, so scores are comparable over time.
- **Automated regression checks**: run your evaluation metrics (BLEU/ROUGE/LLM-as-judge) on every model or prompt change, and flag any DROP versus the current production baseline before it ships — this is the generation-quality equivalent of a CI test suite.
- **Ongoing production monitoring**: sample real production outputs regularly and run them through LLM-as-judge or targeted human review, since your fixed eval set can't anticipate every real-world input your users will actually send.


In [ ]:
# 📋 COPY-PASTE TEMPLATE - a minimal regression-testing pattern

def run_evaluation_suite(model_outputs, reference_outputs):
    """
    Computes BLEU and ROUGE-L for every (output, reference) pair, and
    returns the AVERAGE across the whole evaluation set - the kind of
    single summary number you'd track over time / put in a dashboard.
    """
    bleu_scores = []
    rouge_scores = []

    for output_text, reference_text in zip(model_outputs, reference_outputs):
        output_tokens = output_text.lower().split()
        reference_tokens = reference_text.lower().split()

        bleu_scores.append(compute_bleu(output_tokens, reference_tokens))
        rouge_scores.append(rouge_l(output_tokens, reference_tokens))

    return {
        "average_bleu": sum(bleu_scores) / len(bleu_scores),
        "average_rouge_l": sum(rouge_scores) / len(rouge_scores),
    }

# A tiny example evaluation set:
example_outputs = ["the cat sat on the mat", "dogs love to play fetch"]
example_references = ["the cat is sitting on the mat", "dogs really love to play fetch"]

results = run_evaluation_suite(example_outputs, example_references)
print(results)

# In a real pipeline: save this dict with a timestamp/model-version tag every
# time you evaluate, and alert if average_bleu or average_rouge_l drops
# below your current production baseline.


## Recap & What's Next

You built a real, working extractive summarizer, understand the encoder-decoder shape behind translation and the two flavors of question answering, and — the core of this module — implemented BLEU and ROUGE completely from scratch, understand exactly what BERTScore and perplexity add on top, and know why LLM-as-judge has become the practical default for anything BLEU/ROUGE can't capture well. That closes the evaluation loop that's been implicit in every module of this course.

### Try this before wrapping up
1. Run the extractive summarizer (Part 1) on an article of your own, and compute ROUGE-1/ROUGE-L against a summary you write by hand.
2. Compute BLEU for a few candidate/reference pairs where the wording differs a lot but the MEANING is the same — watch the score stay low, and connect that back to why BERTScore exists.
3. Write your own LLM-as-judge prompt (Part 8) for a task you actually care about, with clear, specific scoring criteria — vague criteria produce vague, inconsistent judge scores.

### Where this course goes from here
You now have the full pipeline, start to finish: acquiring data, representing text, embeddings, classical ML, RNNs/LSTMs, Transformers, fine-tuning, RAG, and evaluation. From here, the path forward is less linear and more about depth in the direction that matters to you:
- **Multimodal NLP** — models that combine text with images/audio/video
- **Speech**: deeper ASR/TTS beyond Module 1's transcription basics
- **MLOps for NLP**: CI/CD for models, feature stores, experiment tracking (MLflow/Weights & Biases) at a deeper level than this course's production notes
- **A specific domain**: legal, medical, code, or another vertical, where domain-specific data and evaluation matter as much as the general techniques
Pick whichever pulls at you most, and you now have the foundation to go build it.
